# Cell 21: Testing accuracy of Model A vs Model E based on rare-subgroup image

In [ ]:
# Systematic rare-subgroup evaluation: Model A vs Model E
rare_subgroups_df = pd.read_csv(OUTPUT_DIR / "rare_subgroups_cgan_targets.csv")
rare_keys = set(
    rare_subgroups_df["dx"] + "|" +
    rare_subgroups_df["age_group"] + "|" +
    rare_subgroups_df["sex"] + "|" +
    rare_subgroups_df["loc_zone"]
)

df_test_copy = df_test.reset_index(drop=True).copy()
df_test_copy["subgroup_key"] = (
    df_test_copy["dx"] + "|" + df_test_copy["age_group"] + "|" +
    df_test_copy["sex"] + "|" + df_test_copy["loc_zone"]
)
df_rare_test = df_test_copy[df_test_copy["subgroup_key"].isin(rare_keys)]

print(f"Rare-subgroup test images: {len(df_rare_test)}")

y_true_rare, y_pred_A_rare, _ = get_predictions(model_a, df_rare_test)
y_true_rare, y_pred_E_rare, _ = get_predictions(model_e, df_rare_test)

acc_A_rare = (y_pred_A_rare == y_true_rare).mean()
acc_E_rare = (y_pred_E_rare == y_true_rare).mean()

print(f"Model A accuracy on RARE subgroups only: {acc_A_rare:.4f}")
print(f"Model E accuracy on RARE subgroups only: {acc_E_rare:.4f}")

# Cell 22: McNemar's test (c must be bigger than b and p < 0.05)

In [ ]:
from scipy.stats import chi2

def mcnemar_test(y_true, y_pred_1, y_pred_2):
    correct_1 = (y_pred_1 == y_true)
    correct_2 = (y_pred_2 == y_true)

    b = int(((correct_1) & (~correct_2)).sum())   # A right, E wrong
    c = int(((~correct_1) & (correct_2)).sum())   # A wrong, E right

    chi_sq = (abs(b - c) - 1)**2 / (b + c) if (b + c) > 0 else 0
    p_value = 1 - chi2.cdf(chi_sq, df=1)

    print(f"  A-right/E-wrong: {b}  |  A-wrong/E-right: {c}")
    print(f"  Chi-squared: {chi_sq:.4f}  |  p-value: {p_value:.4f}")
    print(f"  {'SIGNIFICANT (p<0.05)' if p_value < 0.05 else 'Not significant'}")
    return chi_sq, p_value

print("McNemar's test — Model A vs Model E on RARE subgroup images:")
mcnemar_test(y_true_rare, y_pred_A_rare, y_pred_E_rare)

# Cell 23: Comparison metrics (Worst-group table)

In [ ]:
def get_worst_group_metrics(all_results, models_dict, demo_col="age_group"):
    rows = []
    for mname in models_dict.keys():
        fm = all_results[mname]["fairness"].get(demo_col, {})
        tpr_dict = fm.get("TPR_per_group", {})
        valid_tprs = {k: v for k, v in tpr_dict.items() if not np.isnan(v)}

        if valid_tprs:
            worst_group = min(valid_tprs, key=valid_tprs.get)
            worst_tpr   = valid_tprs[worst_group]
            best_tpr    = max(valid_tprs.values())
            spread      = np.std(list(valid_tprs.values()))
        else:
            worst_group, worst_tpr, best_tpr, spread = None, np.nan, np.nan, np.nan

        rows.append({
            "Model": MODEL_STYLE[mname]["display"].replace("\n"," "),
            "Worst-Group": worst_group,
            "Worst-Group TPR": round(worst_tpr, 4),
            "Best-Group TPR": round(best_tpr, 4),
            "TPR Std Dev (spread)": round(spread, 4),
        })
    return pd.DataFrame(rows).set_index("Model")

worst_group_table = get_worst_group_metrics(all_results, models_dict, "age_group")
print(worst_group_table)
worst_group_table.to_csv(RESULTS_DIR / "worst_group_metrics.csv")